# 03 — Silver transformation

**Purpose:** turn the raw bronze tables into one clean, typed table with real column names, ready for analysis and modeling.

**Reads from:**
- `kitsune_dev.bronze.syn_dos_features_raw` (115 feature columns as text, `_c0` to `_c114`)
- `kitsune_dev.bronze.syn_dos_labels_raw` (label per packet)

**Where the logic lives:** the transformation functions are in `src/kitsune/` (Python modules), not in this notebook.
This notebook only **orchestrates**: it imports those functions, runs them in order and writes the result.

| File | What it does |
|---|---|
| `src/kitsune/feature_names.py` | Generates the 115 real column names (e.g. `_c36` → `hh_l5_pcc`) |
| `src/kitsune/silver.py` | Casts types, joins features with labels, runs quality checks |

**Writes to:** `kitsune_dev.silver.syn_dos_packets` (managed Delta, liquid clustering on `_row_number`, one row per packet: `_row_number` + 115 features + `label`).

**Steps:** config → read bronze → transform → validate → write → post-write check.

**Status:** complete. Runs end-to-end and passes all quality checks (2,771,276 rows).

## Setup: reload modules automatically

**What:** turn on *autoreload*.

**Why:** Python reads a `.py` file only the **first time** you import it, and then keeps that version in memory. If you edit `feature_names.py` afterwards, the notebook would keep using the old version until you restart Python.
With autoreload on, Python checks for changes before running each cell and reloads any module that changed.

In [0]:
%load_ext autoreload
%autoreload 2

## Config

**What:** the names of every table this notebook reads and writes, in one place.

**Why:** if we change environment (dev → prod), we only edit `CATALOG` here, not every cell.

**Result:** three full table names (`catalog.schema.table`).

In [0]:
# Pipeline config: change CATALOG to run against another environment
CATALOG = "kitsune_dev"

BRONZE_FEATURES = f"{CATALOG}.bronze.syn_dos_features_raw"
BRONZE_LABELS = f"{CATALOG}.bronze.syn_dos_labels_raw"
SILVER_PACKETS = f"{CATALOG}.silver.syn_dos_packets"

print(BRONZE_FEATURES)
print(BRONZE_LABELS)
print(SILVER_PACKETS)

kitsune_dev.bronze.syn_dos_features_raw
kitsune_dev.bronze.syn_dos_labels_raw
kitsune_dev.silver.syn_dos_packets


## Read bronze and transform

**What:** read both bronze tables, cast and rename the features, clean the labels, and join them by `_row_number`.

**Why:** the logic lives in `src/kitsune/silver.py`; this notebook only calls it in order.

**Result:** a DataFrame `packets` with 117 columns. Nothing is computed yet (Spark is lazy): the work happens when we validate and write.

In [0]:
from src.kitsune.silver import clean_features, clean_labels, join_features_labels

features_raw = spark.table(BRONZE_FEATURES)
labels_raw = spark.table(BRONZE_LABELS)

features = clean_features(features_raw)
labels = clean_labels(labels_raw)
packets = join_features_labels(features, labels)

print(len(packets.columns))

117


## Validate (quality gate)

**What:** check row count, nulls, NaN/inf in the features, and that labels are only 0 or 1.

**Why:** validate **before** writing, so bad data never reaches the silver table. If a rule fails, a `ValueError` stops the notebook here.

**Result:** "Validation passed" or an error saying which rule failed.

In [0]:
from src.kitsune.silver import validate_silver

expected_rows = features_raw.count()

validate_silver(packets, expected_rows)
print(f"Validation passed: {expected_rows} rows")

Validation passed: 2771276 rows


## Write silver table

**What:** save `packets` as a managed Delta table, with liquid clustering on `_row_number`.

**Why:** `overwrite` makes the notebook idempotent (re-running gives the same table, no duplicates). Clustering speeds up filters by row range, like selecting the first N packets for training.

**Result:** the table `kitsune_dev.silver.syn_dos_packets`.

In [0]:
(
    packets.write
    .format("delta")
    .mode("overwrite")
    .clusterBy("_row_number")
    .saveAsTable(SILVER_PACKETS)
)

print(f"Wrote {SILVER_PACKETS}")

Wrote kitsune_dev.silver.syn_dos_packets


## Post-write check

**What:** read the silver table back and count its rows.

**Why:** validation checked the DataFrame, not the table. This catches problems in the write itself, like accidental duplicates if someone changes `overwrite` to `append`.

**Result:** "Post-write check passed" or a `ValueError`.

In [0]:
silver_rows = spark.table(SILVER_PACKETS).count()

if silver_rows != expected_rows:
    raise ValueError(f"Silver has {silver_rows} rows, expected {expected_rows}")

print(f"Post-write check passed: {silver_rows} rows")

Post-write check passed: 2771276 rows


## Table comment and tags

**What:** add a description and tags to the silver table in Unity Catalog.

**Why:** the comment explains the table to people and to Genie; tags (`layer`, `dataset`, `contains_pii`) make it searchable and ready for governance. We use `spark.sql(f"...")` instead of `%sql` so the table name comes from the config. Re-applied after every write, because `overwrite` can reset them.

**Result:** comment and tags visible in Catalog Explorer.

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {SILVER_PACKETS} IS
    'Clean Kitsune SYN DoS packets, one row per packet in chronological order. 115 typed traffic features with descriptive names, joined with the label by _row_number. Validated: no nulls, no NaN/inf, label in (0, 1).'
""")

spark.sql(f"""
    ALTER TABLE {SILVER_PACKETS}
    SET TAGS ('layer' = 'silver', 'source' = 'uci_kitsune', 'dataset' = 'syn_dos', 'contains_pii' = 'false')
""")

print(f"Comment and tags applied to {SILVER_PACKETS}")

Comment and tags applied to kitsune_dev.silver.syn_dos_packets
